In [ ]:
import pandas as pd
import altair as alt
import numpy as np
import os
import scipy.stats as stats

In [ ]:
input_directory = '/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/vampseq_data_for_qc'

files = os.listdir(input_directory)
files = ["./Data/vampseq_data_for_qc/" + file for file in files]

print(files)
alt.data_transformers.disable_max_rows()

In [ ]:
# Scatter plot helper function

def corr_scatter(df, rep1, rep2, gene):

    rep1_max = df.max(axis = 0)[rep1]
    rep1_min = df.min(axis = 0)[rep1]
    rep2_max = df.max(axis = 0)[rep2]
    rep2_min = df.min(axis = 0)[rep2]

    x_max = rep1_max * 1.05
    y_max = rep2_max * 1.05
    y_min = rep2_min * 1.05

    scatter = alt.Chart(df).mark_circle().encode(
        x = alt.X(f'{rep1}:Q',
                  scale = alt.Scale(0, x_max)
                  ),
        y = alt.Y(f'{rep2}:Q',
                  scale = alt.Scale(y_min, y_max)
                  )
    )

    corr,_=stats.pearsonr(df[rep1], df[rep2])

    r_text = alt.Chart(pd.DataFrame({
        rep1: [2 * 0.97],
        rep2: [0],
        'text': [f'r = {corr:.3f}']
    })).mark_text(
        align='right',
        baseline='bottom',
        fontSize=18,
        fontWeight='bold',
        color='black'
    ).encode(
        x = alt.X(f'{rep1}:Q',
                  scale = alt.Scale(0, x_max)),
        y = alt.Y(f'{rep2}:Q',
                  scale = alt.Scale(y_min, y_max)
                  ),
        text='text:N'
    )

    scatter = (scatter + r_text + scatter.transform_regression(rep1, rep2).mark_line(color = 'red')).properties(title = gene).resolve_scale(x = 'shared', y = 'shared')

    return scatter

# G6PD correlation

In [ ]:
g6pd_path = files[2]
g6pd_df = pd.read_csv(g6pd_path)

g6pd_df = g6pd_df[~g6pd_df['standard_error'].isin([0, 'NA'])].copy()

g6pd_df = g6pd_df.dropna(how = 'any')
g6pd_df.head()

In [ ]:
g6pd_scatter = corr_scatter(g6pd_df, 'rep1_score', 'rep3_score', 'G6PD')

g6pd_scatter.display()

# FIX Correlation

In [ ]:
fix_path = files[0]

fix_df = pd.read_csv(fix_path)
fix_df = fix_df[~fix_df['N_replicates'].isin([9, 0, 1])].copy()
fix_df.head()

In [ ]:
tiles = ['Tile1', 'Tile2', 'Tile3']
fix_dfs = []